##### DATA 620 - Project 3
## Name Gender Classification Using NLP (Natural Language Processing)
**Author**
- Barakat Adigun

## YOUTUBE RECORDING VIDEO LINK

https://youtu.be/dZWHmDdvLSE

## Introduction

The objective of this project is to build a machine learning classifier that predicts whether a person's name is male or female using the NLTK Names Corpus. Following the instructions from Chapter 6 of *Natural Language Processing with Python*, I begin with the example classifier that uses only the last letter of a name. I then incrementally improve the classifier by engineering additional features and comparing multiple classification algorithms.

To avoid overfitting, the dataset is divided into three subsets:

- **Training Set:** Used to train the classifiers.
- **Development Test (Dev-Test) Set:** Used to evaluate improvements while developing the model.
- **Test Set:** Used only once at the end to evaluate the final classifier on unseen data.

Using separate datasets allows me to make improvements objectively while obtaining an unbiased estimate of the model's performance.

In [35]:
import nltk
import random
from nltk.corpus import names
from nltk import NaiveBayesClassifier
from nltk.classify import accuracy

nltk.download('names')

[nltk_data] Downloading package names to /Users/baadigun/nltk_data...
[nltk_data]   Package names is already up-to-date!


True

## Step 1: Load and Prepare the Dataset

The NLTK Names Corpus contains two files: one containing male names and another containing female names. I combine these files into a single labeled dataset where every name is paired with its corresponding gender label.

After combining the datasets, I shuffle the names randomly so that male and female names are mixed together before creating the training, development, and testing datasets.

In [36]:
labeled_names = (
    [(name, 'male') for name in names.words('male.txt')] +
    [(name, 'female') for name in names.words('female.txt')]
)

random.seed(42)
random.shuffle(labeled_names)

print("Total Names:", len(labeled_names))

Total Names: 7944


## Step 2: Split the Dataset

Following the project instructions, I divide the dataset into three subsets:

- 500 names for the final test set.
- 500 names for the development test set.
- The remaining names for the training set.

The development test set is used while improving the classifier, while the test set remains untouched until the very end. This approach helps prevent overfitting and provides a more reliable estimate of the classifier's true performance.

In [37]:
test_names = labeled_names[:500]
devtest_names = labeled_names[500:1000]
train_names = labeled_names[1000:]

print("Training Set:", len(train_names))
print("Dev-Test Set:", len(devtest_names))
print("Test Set:", len(test_names))

Training Set: 6944
Dev-Test Set: 500
Test Set: 500


## Step 3: Build the Baseline Classifier

I begin with the example classifier presented in the textbook. This classifier uses only the final letter of each name as its feature.

Although this is a very simple model, it provides a useful baseline for measuring whether additional feature engineering improves prediction accuracy.

In [38]:
def gender_features_basic(name):
    return {
        'last_letter': name[-1].lower()
    }

train_set_basic = [
    (gender_features_basic(name), gender)
    for (name, gender) in train_names
]

devtest_set_basic = [
    (gender_features_basic(name), gender)
    for (name, gender) in devtest_names
]

classifier_basic = NaiveBayesClassifier.train(train_set_basic)

print("Baseline Dev-Test Accuracy:",
      accuracy(classifier_basic, devtest_set_basic))

Baseline Dev-Test Accuracy: 0.754


## Step 4: Improve the Feature Set

The last letter alone cannot capture every naming pattern. To improve classification accuracy, I engineer additional features that may contain useful information.

The new features include:

- First letter
- Last letter
- First two letters
- Last two letters
- Last three letters
- Name length
- Number of vowels
- Whether the name ends with a vowel

These features allow the classifier to learn more detailed spelling patterns associated with male and female names.

In [39]:
def gender_features_improved(name):

    name = name.lower()

    return {
        'first_letter': name[0],
        'last_letter': name[-1],
        'first_two': name[:2],
        'last_two': name[-2:],
        'last_three': name[-3:],
        'name_length': len(name),
        'count_a': name.count('a'),
        'count_e': name.count('e'),
        'count_i': name.count('i'),
        'count_o': name.count('o'),
        'count_u': name.count('u'),
        'ends_with_vowel': name[-1] in 'aeiou'
    }

train_set = [
    (gender_features_improved(name), gender)
    for (name, gender) in train_names
]

devtest_set = [
    (gender_features_improved(name), gender)
    for (name, gender) in devtest_names
]

classifier_nb = NaiveBayesClassifier.train(train_set)

print("Improved Dev-Test Accuracy:",
      accuracy(classifier_nb, devtest_set))

Improved Dev-Test Accuracy: 0.804


## Step 5: Analyze Classification Errors

Rather than looking only at accuracy, it is useful to inspect the names that are classified incorrectly. Error analysis helps identify weaknesses in the model and may suggest additional features that could improve performance.

Many errors occur because some names are uncommon, are used for both genders, or do not follow typical spelling patterns.

In [40]:
errors = []

for name, gender in devtest_names:

    prediction = classifier_nb.classify(
        gender_features_improved(name)
    )

    if prediction != gender:
        errors.append((gender, prediction, name))

print("Total Errors:", len(errors))

errors[:20]

Total Errors: 98


[('female', 'male', 'Cherey'),
 ('female', 'male', 'Marlo'),
 ('female', 'male', 'Hildagard'),
 ('female', 'male', 'Melicent'),
 ('female', 'male', 'Moll'),
 ('male', 'female', 'Georgie'),
 ('female', 'male', 'Robyn'),
 ('female', 'male', 'Charil'),
 ('male', 'female', 'Etienne'),
 ('female', 'male', 'Joycelin'),
 ('male', 'female', 'Vinny'),
 ('female', 'male', 'Yehudit'),
 ('female', 'male', 'Ivy'),
 ('female', 'male', 'Trudy'),
 ('female', 'male', 'Margalo'),
 ('male', 'female', 'Andre'),
 ('male', 'female', 'Benji'),
 ('male', 'female', 'Martainn'),
 ('female', 'male', 'Pris'),
 ('female', 'male', 'Trish')]

## Step 6: Compare Multiple Classifiers

Feature engineering is only one way to improve performance. Different machine learning algorithms may also perform differently.

To compare algorithms fairly, I train three classifiers using the same feature set:

- Naive Bayes
- Decision Tree
- Maximum Entropy

Each classifier is evaluated using the same development test set.

In [41]:
from nltk.classify import DecisionTreeClassifier
from nltk.classify import MaxentClassifier

classifier_dt = DecisionTreeClassifier.train(train_set)

classifier_me = MaxentClassifier.train(
    train_set,
    max_iter=10,
    trace=0
)

print("Naive Bayes:",
      accuracy(classifier_nb, devtest_set))

print("Decision Tree:",
      accuracy(classifier_dt, devtest_set))

print("Maximum Entropy:",
      accuracy(classifier_me, devtest_set))

Naive Bayes: 0.804
Decision Tree: 0.734
Maximum Entropy: 0.808


## Step 7: Final Evaluation on the Test Set

After selecting the best-performing classifier using the development test set, I evaluate the classifiers one final time using the reserved test set.

Because the test set was never used during model development, it provides an unbiased estimate of how well the classifier generalizes to new names.

In [42]:
test_set = [
    (gender_features_improved(name), gender)
    for (name, gender) in test_names
]

print("Naive Bayes Test Accuracy:",
      accuracy(classifier_nb, test_set))

print("Decision Tree Test Accuracy:",
      accuracy(classifier_dt, test_set))

print("Maximum Entropy Test Accuracy:",
      accuracy(classifier_me, test_set))

Naive Bayes Test Accuracy: 0.782
Decision Tree Test Accuracy: 0.712
Maximum Entropy Test Accuracy: 0.81


In [43]:
import pandas as pd

results = pd.DataFrame({
    "Classifier": [
        "Naive Bayes",
        "Decision Tree",
        "Maximum Entropy"
    ],
    "Dev-Test Accuracy": [
        accuracy(classifier_nb, devtest_set),
        accuracy(classifier_dt, devtest_set),
        accuracy(classifier_me, devtest_set)
    ],
    "Test Accuracy": [
        accuracy(classifier_nb, test_set),
        accuracy(classifier_dt, test_set),
        accuracy(classifier_me, test_set)
    ]
})

results

,Classifier,Dev-Test Accuracy,Test Accuracy
0,Naive Bayes,0.804,0.782
1,Decision Tree,0.734,0.712
2,Maximum Entropy,0.808,0.810


# Conclusion

# Conclusion

This project explored several machine learning classifiers for predicting gender from names in the NLTK Names Corpus. I first implemented the baseline classifier using only the last letter of each name and then improved the model by adding additional spelling-based features such as the first letter, prefixes, suffixes, name length, vowel counts, and whether the name ends with a vowel.

Three classifiers were evaluated: Naive Bayes, Decision Tree, and Maximum Entropy. The Maximum Entropy classifier produced the best overall performance, achieving a development test accuracy of 0.808 and a final test accuracy of 0.810. The Naive Bayes classifier also performed well, while the Decision Tree classifier had the lowest accuracy.
In conclusion, the additional feature engineering improved the classifier compared with the baseline model and demonstrated that multiple spelling characteristics provide more information than using only the final letter of a name.

### Question 1

### How does the performance on the test set compare to the performance on the dev-test set?

The development test and test accuracies were very similar for all three classifiers. The Maximum Entropy classifier achieved the highest performance, improving slightly from 0.808 on the development test set to 0.810 on the test set. Naive Bayes decreased slightly from 0.804 to 0.782, while the Decision Tree classifier decreased from 0.734 to 0.712.

### Question 2

### Is this what you would expect?

Yes. Small differences between the development test and test accuracies are expected because the datasets contain different names. Since the test set was not used during model development, it provides an unbiased estimate of the classifier's ability to generalize to new data. The similar accuracies indicate that the classifiers did not overfit the development test set.